In [1]:
#@title Imports & File Paths

import json
import re
import numpy as np
from pathlib import Path

TRAINED_FILE   = "/content/medgemma_responses_WITH_QWEN_JUDGMENT_trained.jsonl"
UNTRAINED_FILE = "/content/medgemma_responses_WITH_QWEN_JUDGMENT_untrained.jsonl"

In [2]:
#@title Load Result Files

def load_jsonl(path):
    try:
        data = [json.loads(line) for line in open(path, "r", encoding="utf-8") if line.strip()]
        print(f"Loaded {len(data):4d} entries ← {Path(path).name}")
        return data
    except FileNotFoundError:
        print(f"NOT FOUND: {path}")
        return []

trained_data   = load_jsonl(TRAINED_FILE)
untrained_data = load_jsonl(UNTRAINED_FILE)

print(f"\nTotal → Trained: {len(trained_data)}, Untrained: {len(untrained_data)}")

Loaded  743 entries ← medgemma_responses_WITH_QWEN_JUDGMENT_trained.jsonl
Loaded  743 entries ← medgemma_responses_WITH_QWEN_JUDGMENT_untrained.jsonl

Total → Trained: 743, Untrained: 743


In [3]:
#@title Helper Functions

def normalize(text):
    return re.sub(r'\s+', ' ', text.strip().lower())

def longest_common_substring_words(s1, s2):
    w1 = normalize(s1).split()
    w2 = normalize(s2).split()
    if not w1 or not w2: return 0
    dp = [[0] * (len(w2)+1) for _ in range(len(w1)+1)]
    max_len = 0
    for i in range(1, len(w1)+1):
        for j in range(1, len(w2)+1):
            if w1[i-1] == w2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
                max_len = max(max_len, dp[i][j])
    return max_len

def copy_ratio(orig, gen):
    o_words = set(normalize(orig).split())
    g_words = normalize(gen).split()
    return sum(w in o_words for w in g_words) / len(g_words) if g_words else 0.0

In [4]:
#@title Analysis

def analyze(data, name):
    if not data:
        return None

    # Judgment stats
    valid = [e for e in data if e.get("judgment", {}).get("winner") in ["original", "generated", "tie"]]
    total_valid = len(valid)
    orig_wins = sum(1 for e in valid if e["judgment"].get("winner") == "original")
    gen_wins  = sum(1 for e in valid if e["judgment"].get("winner") == "generated")
    ties = total_valid - orig_wins - gen_wins
    decisive = orig_wins + gen_wins
    gen_win_rate = gen_wins / decisive * 100 if decisive > 0 else 0

    avg_orig_score = np.mean([e["judgment"].get("original_score", 0) or 0 for e in valid])
    avg_gen_score  = np.mean([e["judgment"].get("generated_score", 0) or 0 for e in valid])

    # Diversity stats
    div = []
    for e in data:
        if "original" not in e or "generated" not in e: continue
        op = e["original"]["prompt"]
        gp = e["generated"]["prompt"]
        div.append({
            "exact_leak": op in gp,
            "lcs": longest_common_substring_words(op, gp),
            "copy_ratio": copy_ratio(op, gp)
        })

    exact_leaks = sum(d["exact_leak"] for d in div)
    avg_lcs = np.mean([d["lcs"] for d in div])
    avg_copy = np.mean([d["copy_ratio"] for d in div]) * 100

    return {
        "name": name,
        "total": len(data),
        "valid": total_valid,
        "orig_wins": orig_wins,
        "gen_wins": gen_wins,
        "ties": ties,
        "gen_win_rate": gen_win_rate,
        "avg_orig_score": avg_orig_score,
        "avg_gen_score": avg_gen_score,
        "exact_leak_pct": exact_leaks/len(div)*100 if div else 0,
        "avg_lcs": avg_lcs,
        "avg_copy_ratio": avg_copy
    }

trained_result   = analyze(trained_data,   "Trained")
untrained_result = analyze(untrained_data, "Untrained")

print("Analysis complete!")

Analysis complete!


In [5]:
#@title COMPARISON TABLE

t = trained_result
u = untrained_result

if not t or not u:
    print("One of the files is empty — cannot compare")
else:
    print("=" * 95)
    print("                TRAINED vs UNTRAINED — FULL COMPARISON")
    print("=" * 95)
    print(f"{'Metric':<44} {'Trained':>14} {'Untrained':>14} {'Winner':>12}")
    print("-" * 95)
    print(f"{'Total entries':<44} {t['total']:14,} {u['total']:14,}")
    print(f"{'Valid judgments':<44} {t['valid']:14,} {u['valid']:14,}")
    print(f"{'Generated wins (with ties)':<44} {t['gen_wins']/t['valid']*100:13.1f}% {u['gen_wins']/u['valid']*100:13.1f}% {'← TRAINED':>12}")
    print(f"{'Win rate (excluding ties)':<44} {t['gen_win_rate']:13.1f}% {u['gen_win_rate']:13.1f}% {'← TRAINED':>12}")
    print(f"{'Avg score — Generated (/10)':<44} {t['avg_gen_score']:14.2f} {u['avg_gen_score']:14.2f} {'← TRAINED':>12}")
    print(f"{'Exact prompt leaks':<44} {t['exact_leak_pct']:13.2f}% {u['exact_leak_pct']:13.2f}% {'← TRAINED':>12}")
    print(f"{'Avg longest copied chunk (words)':<44} {t['avg_lcs']:14.2f} {u['avg_lcs']:14.2f} {'← TRAINED':>12}")
    print(f"{'Avg Copy Ratio (%)':<44} {t['avg_copy_ratio']:13.2f}% {u['avg_copy_ratio']:13.2f}% {'← TRAINED':>12}")
    print("=" * 95)

                TRAINED vs UNTRAINED — FULL COMPARISON
Metric                                              Trained      Untrained       Winner
-----------------------------------------------------------------------------------------------
Total entries                                           743            743
Valid judgments                                         743            743
Generated wins (with ties)                            65.4%          56.9%    ← TRAINED
Win rate (excluding ties)                             65.4%          56.9%    ← TRAINED
Avg score — Generated (/10)                            8.23           8.07    ← TRAINED
Exact prompt leaks                                    0.00%        100.00%    ← TRAINED
Avg longest copied chunk (words)                       4.40          22.51    ← TRAINED
Avg Copy Ratio (%)                                   32.73%         39.54%    ← TRAINED


In [12]:
#@title Show n Examples

n = 1

print("═" * 110)
print(f"        SHOWING {n} FULL EXAMPLES — TRAINED vs UNTRAINED (with responses)")
print("═" * 110)

# Find all cases where the TRAINED model wins
win_indices = []
for i in range(len(trained_data)):
    if trained_data[i].get("judgment", {}).get("winner") == "generated":
        win_indices.append(i)

if not win_indices:
    print("No wins for trained model found!")
else:
    import random
    random.seed(42)  # reproducible
    selected = random.sample(win_indices, min(n, len(win_indices)))

    for count, idx in enumerate(selected, 1):
        t = trained_data[idx]
        u = untrained_data[idx] if idx < len(untrained_data) else None

        qid = t.get("qid", "N/A")
        judgment = t.get("judgment", {})
        winner = judgment.get("winner", "N/A").upper()
        score  = judgment.get("generated_score", "?")
        feedback = judgment.get("feedback", "No feedback")

        # Prompts
        orig_prompt      = t["original"]["prompt"]
        untrained_prompt = u["generated"]["prompt"] if u else "[MISSING]"
        trained_prompt   = t["generated"]["prompt"]

        # Responses
        orig_response      = t["original"]["response"]
        untrained_response = u["generated"]["response"] if u else "[MISSING UNTRAINED RESPONSE]"
        trained_response   = t["generated"]["response"]

        print(f"\nEXAMPLE {count}/{len(selected)} | qid: {qid} | JUDGE PREFERS → {winner} ({score}/10)")
        print("─" * 110)

        print("ORIGINAL PROMPT:")
        print(orig_prompt.strip())
        print()

        print("UNTRAINED GENERATED PROMPT:")
        print(untrained_prompt.strip())
        print()

        print("TRAINED GENERATED PROMPT:")
        print(trained_prompt.strip())
        print()

        print("UNTRAINED RESPONSE (preview):")
        print(untrained_response[:700].strip() + ("..." if len(untrained_response) > 700 else ""))
        print()

        print("TRAINED RESPONSE (preview):")
        print(trained_response[:700].strip() + ("..." if len(trained_response) > 700 else ""))
        print()

        print("JUDGE FEEDBACK:")
        print(feedback.strip())
        print("\n" + "═" * 110)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
        SHOWING 1 FULL EXAMPLES — TRAINED vs UNTRAINED (with responses)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════

EXAMPLE 1/1 | qid: 3487 | JUDGE PREFERS → GENERATED (9/10)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
ORIGINAL PROMPT:
How does the overuse of antibiotics contribute to the evolution of antibiotic resistance in bacteria and what are the potential consequences of this for human health?

UNTRAINED GENERATED PROMPT:
How does the overuse of antibiotics contribute to the evolution of antibiotic resistance in bacteria and what are the potential consequences of this for human health? Please provide a detailed explanation with multiple steps that illustrate how each step contributes to the development of antibiotic-resistant bacteria.

T